<div dir="rtl" align="right">

# المُرشِّحُ الغاوسيُّ \(Gaussian Filter\)

**مجموعةُ البياناتِ**: PhysioNet Auditory EEG (Abo Alzahab et al., 2021)  
**القنواتُ**: P4, Cz, F8, T7  
**معدّلُ أخذِ العيناتِ**: 200 Hz  
**المُشاركُ**: 1

---

## نظرةٌ عامّةٌ

يَستخدمُ المُرشِّحُ الغاوسيُّ نواةً على شكلِ جرسٍ تَترجّحُ فيها العيّناتُ البعيدةُ أقلّ. كلّما كَبُرَ `sigma`، زادَ التنعيمُ وانتشرَ تأثيرُ كلِّ عيّنةٍ على جيرانٍ أبعدَ.

## المُخرجاتُ المُتوقّعةُ

- sigma=2: تَنعيمٌ خفيفٌ يُحافظُ على التفاصيلِ
- sigma=5: تَنعيمٌ مُعتدلٌ يُزيلُ الضجيجَ السريعَ
- sigma=10: تَنعيمٌ قويٌّ يُنعمُ الإشارةَ بشكلٍ ملحوظٍ

## المُعاملاتُ الأساسيةُ

| المُعاملُ | القيمةُ | المعنى |
| --- | --- | --- |
| القناةُ | P4 | المنطقةُ الجداريةُ |
| معدّلُ الأخذِ | 200 Hz | عيّنةٌ كلَّ 5 ms |
| sigma | 2, 5, 10 | عرضُ النواةِ الغاوسيةِ |
| العيّناتُ المرسومةُ | 5000 | أولُ 25 ثانيةً |

</div>

<div dir="rtl" align="right">

## 1. تثبيتُ المكتباتِ

</div>

In [ ]:
!pip install scipy numpy plotly wfdb


<div dir="rtl" align="right">

## 2. استنساخُ المستودعِ وتنزيلُ بياناتِ مُشاركٍ واحدٍ

نَنزّلُ مُشاركًا واحدًا فقط (`--subjects 1`) لتسريعِ التجربةِ في بيئةِ Colab.

</div>

In [ ]:
import os
if not os.path.exists('python-EEG-Arabic-Resources'):
    !git clone https://github.com/NibrasAz7/python-EEG-Arabic-Resources.git
os.chdir('python-EEG-Arabic-Resources')


In [ ]:
from pathlib import Path
data_dir = Path('data/local')
if not data_dir.exists() or not any(data_dir.glob('*.dat')):
    !python data/download_local.py --output data/local --subjects 1


<div dir="rtl" align="right">

## 3. تحميلُ إشارةِ EEG

نحمّلُ تسجيلَ المُشاركِ 1 في التجربةِ 1، الجلسةِ 2، قناةَ P4 (المنطقةُ الجداريةُ).

</div>

In [ ]:
import numpy as np
from utils.eeg_loader import load_local_eeg

timestamps, eeg_data, ch_names = load_local_eeg(
    data_dir='data/local', subject=1, experiment=1, session=2
)
channel_data = eeg_data[:, 0]  # P4 channel
fs = 200  # Sampling rate (Hz)

print(f'Channels: {ch_names}')
print(f'Signal length: {len(channel_data)} samples ({len(channel_data)/fs:.1f} seconds)')


<div dir="rtl" align="right">

## 4. تطبيقُ المُرشِّحِ

نَستخدمُ `scipy.ndimage.gaussian_filter1d` التي تُطبّقُ نواةً غاوسيةً على بُعدِ واحدٍ. يَتحكمُ `sigma` في عرضِ النواةِ.

</div>

In [ ]:
from scipy.ndimage import gaussian_filter1d

sigmas = [2, 5, 10]
filtered = {}
for s in sigmas:
    filtered[s] = gaussian_filter1d(channel_data, sigma=s)
print(f'Applied Gaussian filter with sigmas: {sigmas}')


<div dir="rtl" align="right">

## 5. رسمٌ تفاعليٌّ

**علامَ تُلاحظُ؟**

- sigma=2: تَبقى التفاصيلُ واضحةً
- sigma=5: يَختفي الضجيجُ السريعُ وتَبقى الإشارةُ مُتماسكةً
- sigma=10: تَنعيمٌ قويٌّ يُخفي التفاصيلَ الدقيقةَ
- قارنْ بين النوافذِ الثلاثِ لرؤيةِ أثرِ `sigma`

</div>

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

n_plot = min(5000, len(channel_data))
t_sec = timestamps[:n_plot] / 1000.0

fig = make_subplots(rows=4, cols=1, shared_xaxes=True,
                    subplot_titles=('Original (P4)',
                                    'Gaussian (sigma=2)',
                                    'Gaussian (sigma=5)',
                                    'Gaussian (sigma=10)'))
fig.add_trace(go.Scatter(x=t_sec, y=channel_data[:n_plot], name='Raw',
                         line=dict(color='gray', width=0.5)), row=1, col=1)
for i, s in enumerate(sigmas, start=2):
    fig.add_trace(go.Scatter(x=t_sec, y=filtered[s][:n_plot],
                             name=f'sigma={s}', line=dict(width=0.5)), row=i, col=1)
fig.update_layout(height=900, title_text='Gaussian Filter - Channel P4',
                  xaxis4_title='Time (s)', showlegend=False)
fig.show()


<div dir="rtl" align="right">

## خلاصةٌ

- يَستخدمُ المُرشِّحُ الغاوسيُّ نواةً على شكلِ جرسٍ تَترجّحُ فيها العيّناتُ البعيدةُ أقلّ
- `sigma` الكبيرُ يُنتجُ تنعيمًا أقوى
- يُحافظُ الغاوسيُّ على شكلِ الإشارةِ أفضلَ من المتوسطِ المتحرّكِ
- يُناسبُ تقليلَ الضجيجِ قبلَ تحليلِ التردداتِ

</div>